# Triangle Shirtwaist Factory — Historical Generative Agent Simulation

**EDUC_5919 — Deep Learning and Transformer Models**

This notebook runs a three-agent generative simulation of the Triangle Shirtwaist Factory on the afternoon of **Saturday, March 25, 1911** — thirty minutes before the fire that killed 146 workers. The agents are:

- **Leonora Russo** (22) — Sicilian immigrant, shirtwaist cutter, supporting her family and saving for her sister's passage.
- **Max Bernstein** (48) — Jewish immigrant floor supervisor and partial owner, a self-made man.
- **Rosa Peretz** (24) — ILGWU organizer, survivor of the 1909 Uprising of the 20,000, banned from the building.

The goal is not to re-create the fire. It is to make visible the structural forces — economic desperation, identity and self-interest, legal permissiveness — that produced the catastrophe through many locally "rational" choices.

The architecture adapts the Stanford generative-agents design (Park et al., 2023) using the OpenAI chat-completions API. Each agent has:

- an **identity stable set (ISS)** — the layered `innate / learned / currently` string used in every prompt,
- **bootstrap memories** — seeded at load into an **`AssociativeMemory`** stream,
- a per-turn **retrieval step** that scores every memory by `recency + relevance + importance` and injects only the top-k into that turn's system prompt,
- **observation writing** — every utterance another agent makes is written into this agent's memory stream with a model-rated importance score, so later turns can retrieve things said earlier in the conversation,
- **reflection** — when accumulated observation-importance crosses a threshold, the agent draws a small number of high-level first-person insights from its recent memories ("I see that this man is not reachable through argument") and writes them back into its memory as new nodes, which then become retrievable themselves.

The conversation runs as scripted turn-taking; each turn is a separate OpenAI chat-completion call, plus embedding calls for every new observation, plus an importance-rating call for every observation, plus reflection calls whenever the trigger fires. See `memory.py` for the retrieval + reflection scoring (a port of Stanford Town's `cognitive_modules/retrieve.py`, `cognitive_modules/reflect.py`, and `memory_structures/associative_memory.py`).

## 1. Install dependencies

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import os
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path

from openai import OpenAI

from memory import AssociativeMemory

MODEL = "gpt-4o"
AGENTS_DIR = Path("agents")
LOGS_DIR = Path("logs")
RETRIEVAL_K = 5  # memories pulled into the prompt per turn

assert os.environ.get("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY in your shell before launching Jupyter: "
    'export OPENAI_API_KEY="sk-..."'
)

## 2. The agent data model

Each agent JSON mirrors Stanford Town's `scratch.json` layering — `innate` (permanent traits), `learned` (stable background), `currently` (present goal) — plus a flat list of `bootstrap_memories` (what the agent knows, has seen, or fears) and `constraints` (explicit behavioral rules baked into the system prompt to keep responses historically grounded).

In [ ]:
@dataclass
class Agent:
    name: str
    first_name: str
    last_name: str
    age: int
    innate: str
    learned: str
    currently: str
    daily_plan_req: str
    bootstrap_memories: list
    constraints: list

def load_agent(path):
    with open(path) as f:
        return Agent(**json.load(f))

agents = {
    "leonora": load_agent(AGENTS_DIR / "leonora_russo.json"),
    "max":     load_agent(AGENTS_DIR / "max_bernstein.json"),
    "rosa":    load_agent(AGENTS_DIR / "rosa_peretz.json"),
}

for key, a in agents.items():
    print(f"{key:8s}  {a.name}, {a.age} — {a.innate}")

## 3. Building each agent's system prompt

`get_str_iss` is a direct port of Stanford Town's `Scratch.get_str_iss()` — the "bare minimum description of the persona that gets used in almost all prompts." `build_system_prompt` wraps the ISS with the scene description, the **top-k retrieved memories** for this turn (not all bootstrap memories — just the ones scored most relevant by the retrieval module), and the agent's constraints, and instructs the model to produce a single short in-character turn.

In [ ]:
def get_str_iss(agent):
    return (
        f"Name: {agent.name}\n"
        f"Age: {agent.age}\n"
        f"Innate traits: {agent.innate}\n"
        f"Learned traits: {agent.learned}\n"
        f"Currently: {agent.currently}\n"
        f"Daily plan requirement: {agent.daily_plan_req}\n"
        f"Current date: Saturday, March 25, 1911\n"
    )

def build_system_prompt(agent, scene_context, retrieved_memories):
    memories = "\n".join(f"- {m}" for m in retrieved_memories)
    constraints = "\n".join(f"- {c}" for c in agent.constraints)
    return f"""You are roleplaying a historical character in a classroom simulation designed to help students see how social, economic, and cultural forces shape individual choices. Stay in character. Do not narrate as a modern observer. Do not break the fourth wall.

# Scene
{scene_context}

# You are {agent.name}.
{get_str_iss(agent)}

# What is in your head right now (the memories most relevant to this moment)
{memories}

# Constraints on your behavior
{constraints}

# How to respond
On each turn you will be shown the conversation so far. Respond with ONLY what {agent.first_name} says or does next — one to three sentences of dialogue, optionally with a brief physical action in *asterisks*. Do not include your name as a speaker label; the simulation adds that. Do not narrate other characters' thoughts or actions. Do not resolve the scene — stay in the moment."""

## 4. One turn = retrieval + one OpenAI API call

For each agent turn we:

1. Build a **retrieval query** from the agent's current goal (`currently`) plus the last couple of utterances they've heard. This is a simplification of Stanford Town's multi-focal-point retrieval — a single concatenated query instead of a list of focal points.
2. Call `memory.retrieve(query, current_turn, k=RETRIEVAL_K)` on the agent's `AssociativeMemory` to score every node by `recency + relevance + importance` and pull the top-k.
3. Send one OpenAI chat-completion with:
   - **system**: identity (ISS) + scene + the **retrieved** memories only + constraints
   - **user**: the running transcript of the conversation so far + "what does \<name\> say next?"

This is the key difference from the pre-retrofit version: the system prompt no longer stuffs in every bootstrap memory. The model only sees the five (by default) memories the retrieval module judges most relevant to this specific moment, which means later turns can retrieve things said *earlier in the conversation* instead of always looking at the same static bootstrap list.

In [ ]:
def format_history(history):
    if not history:
        return "(The conversation has not begun yet. You are about to speak first.)"
    return "\n".join(f"{speaker}: {utterance}" for speaker, utterance in history)

def build_retrieval_query(agent, history):
    """Concatenate the agent's current goal with the last two things they've heard.
    Simpler than Stanford Town's multi-focal-point retrieval — one query per turn."""
    recent = " ".join(u for _, u in history[-2:])
    return f"{agent.currently} {recent}".strip()

def agent_turn(agent, memory, history, turn_num, scene_context, client):
    query = build_retrieval_query(agent, history)
    retrieved_nodes = memory.retrieve(query, current_turn=turn_num, k=RETRIEVAL_K)
    retrieved_strs = [n.content for n in retrieved_nodes]

    user_message = (
        f"Conversation so far on the 9th-floor cutting room:\n\n"
        f"{format_history(history)}\n\n"
        f"What does {agent.first_name} say or do next? Remember: one to three sentences, "
        f"in character, in period voice. No speaker label."
    )
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=300,
        messages=[
            {
                "role": "system",
                "content": build_system_prompt(agent, scene_context, retrieved_strs),
            },
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content.strip(), retrieved_strs

## 5. The scene

The fixed setting: 9th floor cutting room, 4:30 PM, March 25, 1911. Rosa has slipped in through the freight entrance. Max is walking the floor. Leonora is at her cutting table. None of them knows what will happen in thirty minutes.

In [ ]:
SCENE_CONTEXT = """\
It is Saturday, March 25, 1911, at 4:30 PM, in New York City.

The scene is the 9th-floor cutting room of the Triangle Shirtwaist Company, in
the Asch Building at the corner of Washington Place and Greene Street. The floor
is loud with sewing machines on the 8th floor below and the hum of the cutting
tables here. Piles of cotton shirtwaist cuttings are heaped under the tables;
the air is thick with lint. The Washington Place stairwell door is locked, as
it is every shift; the Greene Street freight door is the only open exit. About
four hundred workers are on the floor, mostly young immigrant women. The shift
ends at 5:00 PM.

Leonora Russo is at her cutting table. Rosa Peretz, an ILGWU organizer banned
from the building by name, has slipped in through the freight entrance under a
shawl. She approaches Leonora. Max Bernstein, the floor supervisor and a partial
owner, is walking the floor and will shortly notice Rosa.

None of the three knows that in approximately thirty minutes, a scrap bin on
the 8th floor will ignite and the fire will spread through this building,
killing 146 workers. They are acting in the moment, with the knowledge and
fears available to each of them on this ordinary late-Saturday afternoon.
"""

## 6. Run the interaction

The turn order is scripted: Rosa approaches Leonora first, they exchange a few lines, Max notices Rosa and intervenes, and the three negotiate the confrontation.

Before we start, each agent gets an `AssociativeMemory` seeded with their `bootstrap_memories` (all embedded at load). Then on every turn:

1. **If the agent's reflection trigger has fired**, reflect first — the model reads the agent's recent memories and writes 2 first-person insights back into the memory stream. These happen *before* retrieval, so fresh insights are eligible to be pulled into this turn's system prompt.
2. The speaker's memory stream is **retrieved** against the current retrieval query.
3. The model produces an utterance.
4. **Every other agent** records the utterance as a new observation in their own memory stream, with an importance score rated by a cheap `gpt-4o-mini` call. (The speaker doesn't self-observe; the conversation history already covers that.) Adding the observation decrements that agent's `importance_trigger` by the rated importance; when it hits 0 their reflection fires at the top of their next turn.

Each utterance costs one `gpt-4o` completion + one embedding call per listener + one `gpt-4o-mini` importance rating per listener, plus one extra `gpt-4o` reflection call whenever the trigger fires (~1–2 times per agent in this scene). Expect the cell to take about 90–120 seconds.

In [ ]:
turn_order = [
    "rosa",      # Rosa opens: quiet approach, names the union and the ask
    "leonora",   # Leonora: hesitant, afraid
    "rosa",      # Rosa: specifics — locked doors, sprinklers
    "leonora",   # Leonora: the family, the sister
    "max",       # Max: recognizes Rosa, confronts
    "rosa",      # Rosa: holds ground
    "max",       # Max: the self-made-man frame, orders her out
    "leonora",   # Leonora: caught between, tries to disappear
    "rosa",      # Rosa: last word to Leonora
    "max",       # Max: closes the scene
]

client = OpenAI()

print("[seeding agent memory streams with bootstrap memories…]")
memories = {key: AssociativeMemory(client, a.name) for key, a in agents.items()}
for key, a in agents.items():
    memories[key].seed_bootstrap(a.bootstrap_memories)
    print(f"  {key}: {len(memories[key].nodes)} bootstrap nodes embedded")
print()

history = []
per_turn = []

print("=" * 72)
print(" TRIANGLE SHIRTWAIST FACTORY — 9th FLOOR CUTTING ROOM")
print(" Saturday, March 25, 1911 — 4:30 PM")
print("=" * 72)
print()

for turn_num, agent_key in enumerate(turn_order):
    agent = agents[agent_key]
    memory = memories[agent_key]

    # Reflection fires BEFORE retrieval so fresh insights can be pulled in.
    reflection_log = None
    if memory.should_reflect():
        new_insights = memory.reflect(current_turn=turn_num)
        if new_insights:
            reflection_log = {
                "triggered": True,
                "insights": [n.content for n in new_insights],
            }
            print(f"  [{agent.first_name} reflects]")
            for s in reflection_log["insights"]:
                print(f"    - {s}")

    query = build_retrieval_query(agent, history)
    utterance, retrieved = agent_turn(
        agent, memory, history, turn_num, SCENE_CONTEXT, client
    )
    history.append((agent.first_name, utterance))

    for other_key, other_memory in memories.items():
        if other_key == agent_key:
            continue
        observation = f'{agent.name} said: "{utterance}"'
        other_memory.add_observation(
            content=observation,
            speaker=agent.name,
            turn=turn_num,
            rate_with_model=True,
        )

    per_turn.append({
        "turn": turn_num,
        "speaker": agent.first_name,
        "reflection": reflection_log,
        "retrieval_query": query,
        "retrieved_memories": retrieved,
        "utterance": utterance,
    })

    print(f"{agent.first_name}: {utterance}")
    print()

## 7. Save a JSON log of the run

The log captures the scene, the three agents' ISS fields, the turn order, the transcript, the **per-turn retrieval audit** (query + top-k memory contents that actually went into each prompt), and each agent's **final memory stream**. Embeddings are elided (replaced with `"[1536-dim vector]"`) to keep the file human-readable for grading.

In [ ]:
LOGS_DIR.mkdir(exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = LOGS_DIR / f"run_{ts}.json"
payload = {
    "timestamp": ts,
    "model": MODEL,
    "retrieval_k": RETRIEVAL_K,
    "scene_context": SCENE_CONTEXT,
    "turn_order": turn_order,
    "agents": {key: asdict(a) for key, a in agents.items()},
    "per_turn": per_turn,
    "transcript": [{"speaker": s, "utterance": u} for s, u in history],
    "final_memory_streams": {
        key: memory.dump(include_embeddings=False)
        for key, memory in memories.items()
    },
}
with open(log_path, "w") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print(f"[log saved: {log_path}]")

## 8. What the transcript should reveal

The three-way conversation is designed to make structural forces visible through each agent's dialogue choices. When reading the transcript, look for:

- **Leonora's constraint is economic vulnerability.** She should hesitate, weigh the ask against her family's dependence on her wages, and fear the blacklist. Her agreement or refusal is not a free choice — it is shaped by her father's debt, her sister in Palermo, and her immigration status.
- **Max's constraint is the worldview that made him successful.** He should defend the factory sincerely — not as a villain, but as a man who believes what he says. His framing ("I came with fourteen dollars," "outsiders agitating my girls") is exactly how self-interest becomes invisible to the person holding it.
- **Rosa's constraint is legal and structural.** She should be specific (locked doors, sprinklers, sign the card), not rhetorical. Her urgency and the workers' hesitation together expose *why reform is hard even when danger is visible*.

The 146 deaths that occurred thirty minutes after this scene were not produced by a single villain. They were produced by every actor making choices that were locally rational within their constrained worlds. The point of the simulation is to make those constraints, and the system they compose, legible.

### Inspecting the retrieval and reflection audits

Open the JSON log saved above. The `per_turn` array records, for each turn:

- `retrieval_query` — the string built from the speaker's `currently` + last two utterances
- `retrieved_memories` — the top-k memory contents that were actually injected into that turn's system prompt
- `reflection` — `null` for most turns, but when the reflection trigger has fired this is `{"triggered": true, "insights": [...]}` and the listed insights are the first-person sentences that got written back into the agent's memory stream *before* retrieval for that turn ran

Compare early turns (retrieval pulls mostly from bootstrap) to later turns (retrieval starts pulling in-scene observations and, once reflection has fired, retrieval can surface those insights alongside raw events). The `final_memory_streams` section shows each agent's full post-scene memory: bootstrap seeds + observations + reflection nodes, each with its `source` field and importance score. Reflection nodes are marked `"source": "reflection"` — scanning for those in each agent's stream is the quickest way to see what higher-level understanding each character drew out of the confrontation.